# Python Tutorial — Part 9: Iterators, Generators and Advanced Python Concepts

**Beginner-friendly | Bangla + English | Jupyter Notebook**

এই Part-এ Python-এ data একটির পর একটি কীভাবে access করা হয়, iterator ও generator কীভাবে কাজ করে এবং lazy evaluation, generator expression, `yield from`, `itertools`, decorators ও context managers-এর basic ধারণা শিখব।

## Learning outcomes
- Explain iterable, iterator and iteration.
- Use `iter()` and `next()`.
- Understand `StopIteration`.
- Create a custom iterator using a class.
- Create generators using `yield`.
- Use generator expressions and lazy evaluation.
- Use `yield from` and selected `itertools` functions.
- Understand basic decorators and context managers.
- Complete practical exercises and mini-projects.

## 0. How to use this notebook

1. Open the notebook in Jupyter Notebook, JupyterLab, VS Code or Google Colab.
2. Run each code cell using **Shift + Enter**.
3. Read the explanation before running the example.
4. Predict the output before executing each cell.
5. Change the values and run the cell again.

**Important:** Some examples intentionally show an error. Those cells are marked clearly. Do not worry; the error is part of learning.

## 1. What is Iteration?

**Iteration** means repeating an action or accessing items one by one.

Example: A list contains three names. A `for` loop visits each name one at a time.

In [ ]:
students = ["Rahim", "Karim", "Nila"]

for student in students:
    print(student)

**Output**
```text
Rahim
Karim
Nila
```

Python uses iteration in loops, file reading, strings, lists, tuples and many other situations.

## 2. Iterable vs Iterator

These two terms are related, but they are not the same.

### Iterable
An **iterable** is an object whose items can be accessed one at a time. Examples: list, tuple, string, dictionary, set and range.

### Iterator
An **iterator** is an object that remembers its current position and gives the next item when requested.

সহজভাবে:
- Iterable = যে collection-এর উপর loop চালানো যায়।
- Iterator = যে object একবারে next item দেয় এবং নিজের position মনে রাখে।

In [ ]:
numbers = [10, 20, 30]

# A list is iterable
print(iter(numbers))

`iter(numbers)` list থেকে একটি iterator তৈরি করে। Iterator-এর exact memory representation বা address output-এ ভিন্ন হতে পারে।

## 3. The `iter()` Function

`iter()` একটি iterable object থেকে iterator তৈরি করে।

In [ ]:
fruits = ["apple", "banana", "mango"]

fruit_iterator = iter(fruits)

print(type(fruit_iterator))

**Observe:** `type()` iterator-এর type দেখায়। List এবং list iterator এক জিনিস নয়।

## 4. The `next()` Function

`next()` iterator থেকে পরবর্তী item বের করে।

In [ ]:
colors = ["red", "green", "blue"]
color_iterator = iter(colors)

print(next(color_iterator))
print(next(color_iterator))
print(next(color_iterator))

**Output**
```text
red
green
blue
```

প্রতিবার `next()` call করলে iterator পরবর্তী item-এ চলে যায়।

### 4.1 What happens when there are no more items?

Iterator-এর সব item শেষ হয়ে গেলে আর `next()` call করলে `StopIteration` exception হয়।

In [ ]:
values = [1, 2]
value_iterator = iter(values)

print(next(value_iterator))
print(next(value_iterator))

# This next call raises StopIteration because no item remains.
# Uncomment the following line to observe the exception:
# print(next(value_iterator))

**Output**
```text
1
2
```

`StopIteration` হলো iterator-এর signal যে আর কোনো item বাকি নেই। `for` loop internally এই condition handle করে, তাই সাধারণত loop-এ এটি নিজে দেখতে হয় না।

### 4.2 `next()` with a default value

`next(iterator, default)` ব্যবহার করলে iterator শেষ হলে exception না দিয়ে default value ফেরত দেয়।

In [ ]:
items = ["A", "B"]
item_iterator = iter(items)

print(next(item_iterator, "Finished"))
print(next(item_iterator, "Finished"))
print(next(item_iterator, "Finished"))

**Output**
```text
A
B
Finished
```

## 5. How Does a `for` Loop Work?

একটি `for` loop internally iterable থেকে iterator নেয়, তারপর একের পর এক item নেয়। item শেষ হলে loop বন্ধ হয়।

In [ ]:
numbers = [5, 10, 15]

for number in numbers:
    print(number)

Conceptually, the loop behaves much like:
```python
iterator = iter(numbers)

while True:
    try:
        number = next(iterator)
        print(number)
    except StopIteration:
        break
```
এই code-টি `for` loop-এর ধারণা বোঝানোর জন্য; সাধারণ কাজে `for` loop-ই ব্যবহার করবে।

## 6. Creating a Custom Iterator with a Class

নিজের class-কে iterator বানাতে সাধারণত দুটি special method ব্যবহার করা হয়:
- `__iter__()` — iterator object ফেরত দেয়।
- `__next__()` — পরবর্তী item ফেরত দেয়; item শেষ হলে `StopIteration` raise করে।

In [ ]:
class CountUp:
    def __init__(self, limit):
        self.limit = limit
        self.current = 1

    def __iter__(self):
        return self

    def __next__(self):
        if self.current <= self.limit:
            value = self.current
            self.current += 1
            return value
        raise StopIteration


counter = CountUp(4)

for number in counter:
    print(number)

**Output**
```text
1
2
3
4
```

**Line-by-line explanation**
1. `__init__()` limit এবং starting position সেট করে।
2. `__iter__()` `self` ফেরত দেয়, কারণ object-টিই iterator।
3. `__next__()` দেখে current value limit-এর মধ্যে আছে কি না।
4. থাকলে value ফেরত দিয়ে current এক বাড়ায়।
5. শেষ হলে `StopIteration` raise করে।

## 7. What is a Generator?

A **generator** হলো এমন একটি সহজ উপায় যার মাধ্যমে iterator তৈরি করা যায়। Generator function-এ `yield` ব্যবহার করা হয়।

- `return` function-এর execution শেষ করে।
- `yield` একটি value দেয়, কিন্তু function-এর state মনে রাখে। পরেরবার generator চালালে আগের জায়গা থেকে চলতে থাকে।

সহজভাবে: generator একবারে একটি value তৈরি করে; সব value একসঙ্গে memory-তে তৈরি করে রাখতে হয় না।

In [ ]:
def simple_generator():
    yield 10
    yield 20
    yield 30

result = simple_generator()

print(result)
print(next(result))
print(next(result))
print(next(result))

**Output-এর ধারণা**
```text
<generator object ...>
10
20
30
```
Generator object-এর representation Python version/session অনুযায়ী ভিন্ন হতে পারে।

## 8. Generator Function with a Loop

একটি loop-এর মধ্যে `yield` দিয়ে একাধিক value তৈরি করা যায়।

In [ ]:
def count_up_to(n):
    number = 1
    while number <= n:
        yield number
        number += 1

for value in count_up_to(5):
    print(value)

**Output**
```text
1
2
3
4
5
```

এখানে `count_up_to(5)` call করলে function-এর code সঙ্গে সঙ্গে সব execute না করে একটি generator object তৈরি হয়। Loop যখন value চায়, তখন generator চলতে থাকে।

## 9. `return` vs `yield`

| `return` | `yield` |
|---|---|
| Function থেকে value ফেরত দিয়ে function শেষ করে | Value দেয়, execution pause করে |
| সাধারণ function-এর result ফেরত দেয় | Generator তৈরি করতে ব্যবহৃত হয় |
| একই function call-এর পরে execution শেষ | পরের request-এ আগের অবস্থান থেকে চলে |

Example:

In [ ]:
def normal_function():
    return 1
    return 2

def generator_function():
    yield 1
    yield 2

print(normal_function())
print(list(generator_function()))

**Output**
```text
1
[1, 2]
```
Normal function-এর প্রথম `return`-এর পর function শেষ হয়ে যায়। Generator উভয় `yield` value দিতে পারে।

## 10. Generator Expression

List comprehension-এর মতো দেখতে, কিন্তু square bracket `[]`-এর বদলে parentheses `()` ব্যবহার করলে generator expression তৈরি হয়।

In [ ]:
squares_list = [number * number for number in range(1, 6)]
squares_generator = (number * number for number in range(1, 6))

print("List:", squares_list)
print("Generator values:", list(squares_generator))

**Output**
```text
List: [1, 4, 9, 16, 25]
Generator values: [1, 4, 9, 16, 25]
```

পার্থক্য:
- List comprehension সব result-এর list তৈরি করে।
- Generator expression প্রয়োজন অনুযায়ী একবারে একটি result দেয়।

## 11. Lazy Evaluation

**Lazy evaluation** মানে প্রয়োজন না হওয়া পর্যন্ত value তৈরি বা calculation পিছিয়ে রাখা।

Generator সাধারণত lazyভাবে কাজ করে। বড় data নিয়ে কাজ করার সময় এটি memory সাশ্রয় করতে পারে, কারণ সব result একসঙ্গে list হিসেবে তৈরি করতে হয় না।

In [ ]:
def generate_numbers():
    print("Starting generator")
    yield 1
    print("Making second value")
    yield 2
    print("Making third value")
    yield 3

numbers = generate_numbers()
print("Generator created")

print(next(numbers))
print(next(numbers))

**Output**
```text
Generator created
Starting generator
1
Making second value
2
```

Notice: generator তৈরি করার সময় function-এর ভিতরের print চলে না। `next()` call করলে execution শুরু বা resume হয়।

## 12. `yield from`

`yield from` দিয়ে একটি iterable-এর item-গুলো generator থেকে একটির পর একটি yield করা যায়।

In [ ]:
def all_subjects():
    yield from ["English", "Math", "Python"]

for subject in all_subjects():
    print(subject)

**Output**
```text
English
Math
Python
```

## 13. Useful Functions with Iterables

### 13.1 `enumerate()`

`enumerate()` item-এর সঙ্গে index দেয়।

In [ ]:
subjects = ["Python", "Database", "Networking"]

for index, subject in enumerate(subjects):
    print(index, subject)

**Output**
```text
0 Python
1 Database
2 Networking
```
Starting index পরিবর্তন:

In [ ]:
subjects = ["Python", "Database", "Networking"]

for index, subject in enumerate(subjects, start=1):
    print(index, subject)

### 13.2 `zip()`

`zip()` একাধিক iterable-এর corresponding item একসঙ্গে pair করে।

In [ ]:
names = ["Rahim", "Nila", "Karim"]
marks = [85, 92, 78]

for name, mark in zip(names, marks):
    print(name, mark)

**Output**
```text
Rahim 85
Nila 92
Karim 78
```
সাধারণ `zip()` সবচেয়ে ছোট iterable শেষ হলে থেমে যায়।

### 13.3 `map()`

`map()` একটি function-কে iterable-এর প্রতিটি item-এ apply করে এবং একটি iterator দেয়।

In [ ]:
numbers = [1, 2, 3, 4]

squares = map(lambda x: x * x, numbers)

print(list(squares))

**Output**
```text
[1, 4, 9, 16]
```

### 13.4 `filter()`

`filter()` condition True হয় এমন item-গুলো রাখে।

In [ ]:
numbers = [1, 2, 3, 4, 5, 6]

even_numbers = filter(lambda x: x % 2 == 0, numbers)

print(list(even_numbers))

**Output**
```text
[2, 4, 6]
```

## 14. The `itertools` Module

`itertools` হলো Python standard library-এর module, যেখানে iterator তৈরির জন্য useful tools আছে।

### 14.1 `count()`

একটি সংখ্যা থেকে ধারাবাহিকভাবে count তৈরি করে। এটি infinite iterator, তাই সীমা নির্ধারণ না করে পুরোটা list-এ convert করবে না।

In [ ]:
from itertools import count

counter = count(10, 2)

for _ in range(5):
    print(next(counter))

**Output**
```text
10
12
14
16
18
```

### 14.2 `cycle()`

একটি iterable-এর item বারবার repeat করে। এটিও infinite iterator।

In [ ]:
from itertools import cycle

colors = cycle(["red", "green", "blue"])

for _ in range(7):
    print(next(colors))

**Output**
```text
red
green
blue
red
green
blue
red
```

### 14.3 `islice()`

কোনো iterator-এর নির্দিষ্ট অংশ নিতে `islice()` ব্যবহার করা যায়।

In [ ]:
from itertools import count, islice

numbers = count(1)

first_five = list(islice(numbers, 5))
print(first_five)

**Output**
```text
[1, 2, 3, 4, 5]
```

### 14.4 `chain()`

একাধিক iterable-কে ধারাবাহিকভাবে iterate করতে `chain()` ব্যবহার করা যায়।

In [ ]:
from itertools import chain

group_a = [1, 2, 3]
group_b = [4, 5]

combined = chain(group_a, group_b)
print(list(combined))

**Output**
```text
[1, 2, 3, 4, 5]
```

## 15. Basic Decorators

Decorator হলো এমন একটি function যা অন্য function-কে modify বা extend করে, তার মূল code সরাসরি পরিবর্তন না করেও।

Decorator বুঝতে function-কে অন্য function-এর argument হিসেবে পাঠানোর ধারণা জানা দরকার।

In [ ]:
def greet():
    print("Hello!")

def call_function(function):
    print("Before calling function")
    function()
    print("After calling function")

call_function(greet)

**Output**
```text
Before calling function
Hello!
After calling function
```

### 15.1 Writing a simple decorator

`@decorator_name` syntax দিয়ে decorator apply করা যায়।

In [ ]:
def my_decorator(function):
    def wrapper():
        print("Before function")
        function()
        print("After function")
    return wrapper

@my_decorator
def say_hello():
    print("Hello, Python!")

say_hello()

**Output**
```text
Before function
Hello, Python!
After function
```

`@my_decorator` না লিখে equivalentভাবে `say_hello = my_decorator(say_hello)` করা যায়।

### 15.2 Decorator that accepts arguments

Function-এর arguments ঠিকভাবে pass করার জন্য `*args` এবং `**kwargs` ব্যবহার করা যায়।

In [ ]:
def announce(function):
    def wrapper(*args, **kwargs):
        print("Function is running")
        result = function(*args, **kwargs)
        print("Function finished")
        return result
    return wrapper

@announce
def add(a, b):
    return a + b

answer = add(10, 20)
print("Answer:", answer)

**Output**
```text
Function is running
Function finished
Answer: 30
```

বাস্তব project-এ decorator-এর metadata সংরক্ষণের জন্য `functools.wraps` ব্যবহার করা ভালো।

In [ ]:
from functools import wraps

def announce(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        print("Calling:", function.__name__)
        return function(*args, **kwargs)
    return wrapper

@announce
def multiply(a, b):
    """Multiply two numbers."""
    return a * b

print(multiply(3, 4))
print(multiply.__name__)
print(multiply.__doc__)

## 16. Context Managers — `with` Statement

Context manager কোনো কাজের আগে setup এবং কাজ শেষে cleanup করতে সাহায্য করে। File handling-এ `with open(...)` একটি পরিচিত example।

এখানে একটি simple custom context manager-এর example দেখব।

In [ ]:
class SimpleMessage:
    def __enter__(self):
        print("Entering the context")
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        print("Leaving the context")
        return False

with SimpleMessage():
    print("Inside the context")

**Output**
```text
Entering the context
Inside the context
Leaving the context
```

- `__enter__()` context শুরু হলে চলে।
- `__exit__()` context থেকে বের হলে চলে।
- `return False` exception থাকলে সেটিকে suppress করে না।

## 17. Mini Project 1 — Generate Even Numbers

একটি generator function লিখব যা ২ থেকে নির্দিষ্ট limit পর্যন্ত even number yield করবে।

In [ ]:
def even_numbers(limit):
    number = 2
    while number <= limit:
        yield number
        number += 2

for value in even_numbers(20):
    print(value, end=" ")

**Output**
```text
2 4 6 8 10 12 14 16 18 20
```

**Challenge:** এমন generator তৈরি করো যা ১ থেকে limit পর্যন্ত odd numbers yield করবে।

## 18. Mini Project 2 — Student Marks Generator

একটি marks list থেকে শুধু pass করা student-দের marks generator দিয়ে বের করব। এখানে pass mark ধরা হয়েছে ৪০।

In [ ]:
def passed_marks(marks):
    for mark in marks:
        if mark >= 40:
            yield mark

student_marks = [32, 75, 40, 28, 90, 55]

for mark in passed_marks(student_marks):
    print(mark)

**Output**
```text
75
40
90
55
```

**Challenge:** শুধু ৮০ বা তার বেশি marks yield করো।

## 19. Mini Project 3 — Simple Countdown Generator

Generator দিয়ে countdown তৈরি করব।

In [ ]:
def countdown(start):
    while start >= 1:
        yield start
        start -= 1
    yield "Go!"

for item in countdown(5):
    print(item)

**Output**
```text
5
4
3
2
1
Go!
```

## 20. Practice Questions — Try Yourself

প্রথমে নিজে solve করবে। প্রতিটি question-এর output predict করার চেষ্টা করবে।

1. একটি list থেকে iterator তৈরি করে `next()` দিয়ে প্রথম দুইটি item print করো।
2. `next(iterator, "End")` ব্যবহার করে iterator শেষ হওয়ার পর কী হয় দেখাও।
3. `yield` ব্যবহার করে ১ থেকে ১০ পর্যন্ত number তৈরি করো।
4. একটি generator লিখো যা ১ থেকে n পর্যন্ত square yield করবে।
5. list comprehension এবং generator expression-এর মধ্যে পার্থক্য দেখাও।
6. `enumerate()` দিয়ে subject list-এর index এবং name print করো।
7. `zip()` দিয়ে student name এবং marks pair করো।
8. `filter()` দিয়ে list থেকে odd number বের করো।
9. `itertools.cycle()` দিয়ে তিনটি traffic light color repeat করো।
10. একটি decorator তৈরি করো যা function call-এর আগে "Starting" এবং পরে "Finished" print করবে।

## 21. Practice Solutions

### Solution 1 — Iterator and next

In [ ]:
items = ["A", "B", "C"]
iterator = iter(items)

print(next(iterator))
print(next(iterator))

### Solution 2 — Default value

In [ ]:
items = [10]
iterator = iter(items)

print(next(iterator, "End"))
print(next(iterator, "End"))

### Solution 3 — Generator from 1 to 10

In [ ]:
def numbers_to_ten():
    for number in range(1, 11):
        yield number

print(list(numbers_to_ten()))

### Solution 4 — Squares generator

In [ ]:
def squares(n):
    for number in range(1, n + 1):
        yield number * number

print(list(squares(5)))

### Solution 5 — List and generator expression

In [ ]:
list_result = [x * 2 for x in range(5)]
generator_result = (x * 2 for x in range(5))

print(list_result)
print(list(generator_result))

### Solution 6 — enumerate

In [ ]:
subjects = ["English", "Math", "Python"]

for index, subject in enumerate(subjects, start=1):
    print(index, subject)

### Solution 7 — zip

In [ ]:
names = ["Rina", "Karim"]
marks = [90, 85]

for name, mark in zip(names, marks):
    print(name, mark)

### Solution 8 — filter odd numbers

In [ ]:
numbers = [1, 2, 3, 4, 5, 6]
odd = filter(lambda x: x % 2 != 0, numbers)

print(list(odd))

### Solution 9 — itertools cycle

In [ ]:
from itertools import cycle

lights = cycle(["Red", "Yellow", "Green"])

for _ in range(6):
    print(next(lights))

### Solution 10 — Basic decorator

In [ ]:
def status_decorator(function):
    def wrapper():
        print("Starting")
        function()
        print("Finished")
    return wrapper

@status_decorator
def task():
    print("Doing the task")

task()

## 22. Common Mistakes

1. **Calling `next()` on a list:** A list is iterable, but it is not itself an iterator. Use `next(iter(my_list))` or create an iterator first.
2. **Using an exhausted iterator again:** Once consumed, an iterator normally stays exhausted. Create a new iterator if you need to start again.
3. **Forgetting `yield`:** A function without `yield` is not a generator function.
4. **Converting an infinite iterator to a list:** `list(count())` will not finish. Use a finite loop or `islice()`.
5. **Confusing `return` and `yield`:** `return` ends a function; `yield` pauses a generator and preserves its state.
6. **Decorator wrapper does not return a result:** If the decorated function returns a value, the wrapper should return that value too.

## 23. Part 9 Revision Checklist

- [ ] I can explain iteration.
- [ ] I can distinguish iterable and iterator.
- [ ] I can use `iter()` and `next()`.
- [ ] I understand `StopIteration`.
- [ ] I can create a custom iterator class.
- [ ] I understand generator functions and `yield`.
- [ ] I can create generator expressions.
- [ ] I understand lazy evaluation.
- [ ] I can use `yield from`.
- [ ] I can use `enumerate()`, `zip()`, `map()` and `filter()`.
- [ ] I can use selected functions from `itertools`.
- [ ] I understand the basic idea of decorators.
- [ ] I understand context managers and `with`.

**End of Part 9 — Iterators, Generators and Advanced Python Concepts**

প্রতিটি code cell run করবে, output বোঝার চেষ্টা করবে এবং practice questions নিজে solve করবে।